# 05 — Hybrid RAG

**Priority:** 🟢 Nice-to-have — combines keyword + vector search — good pattern breadth. *If skipped, revisit when:* when pure vector search misses exact-match queries (IDs, part numbers).

```
╔══════════════════════════════════════════════════════════════════╗
║                      5. HYBRID RAG                               ║
║                                                                  ║
║  Documents ──► Chunks ──► Embedding Model ──► Vector Database   ║
║                    │                               │             ║
║                    └──────► LLM Graph Generator ──► Graph DB    ║
║                                                                  ║
║                    ↑ (infographic meaning)                       ║
║   ─────────────────────────────────────────────────────────────  ║
║                    ↓ (industry-standard meaning, also covered)   ║
║                                                                  ║
║  Documents ──► Chunks ──► Embedding Model ──► Vector DB         ║
║                    │                                             ║
║                    └──────────────────────────► BM25 Index      ║
║                                                                  ║
║          Query ──► Dense + Sparse ──► RRF Fusion ──► Top-k      ║
║                                                       │          ║
║  Response ◄── Generative Model ◄── Prompt ◄──────────┘          ║
╚══════════════════════════════════════════════════════════════════╝
```

## Two meanings of "Hybrid RAG"

The term gets used in two different ways:

1. **Infographic meaning (vector + graph)**: Combine the vector store AND graph database from earlier notebooks. This is what the infographic shows. We'll cover this briefly since you built both in NB01 and NB04.

2. **Industry-standard meaning (dense + sparse)**: Combine **dense vector retrieval** (semantic similarity via embeddings) with **sparse keyword retrieval** (BM25/TF-IDF). This is what most practitioners mean by "hybrid RAG". **This is the main focus of this notebook.**

## Why dense + sparse?

Dense embeddings capture *meaning* but struggle with exact strings:
- Part number `HR-REED-UPGRADE` — semantic search might miss this if it's a rare string
- Error code `INC-2024-031` — meaningless semantically but critical for exact lookup
- Technical jargon: `FW-V2-2.3.2`, `HC400-CTRL`, `EtherCAT` — vectors may not encode these precisely

BM25 captures *keywords* but misses synonyms:
- Query "arm heat problem" won't BM25-match "joint 4 thermal issue" well

**Hybrid = both**: Reciprocal Rank Fusion (RRF) combines the ranked lists without needing normalised scores.

## What you'll learn
- Build a BM25 index alongside the vector index
- Implement Reciprocal Rank Fusion (RRF)
- See exactly when exact-match (BM25) beats semantic search and vice versa
- Combine with the graph from NB04 for the maximum-context variant

In [ ]:
import sys; sys.path.insert(0, '..')
import ragkit.config as cfg

cfg.BACKEND = "claude"   # "claude" | "local"
USE_NEO4J = True  # set to False if Neo4j isn't running

print(f"Backend: {cfg.BACKEND}  |  Device: {cfg.DEVICE}")

## Step 1 — Build both indexes

In [ ]:
from ragkit.data import build_chunked_corpus
from ragkit.vectorstore import build_collection
from rank_bm25 import BM25Okapi
import re

texts, metadatas = build_chunked_corpus(chunk_size=200, overlap=40)

# ── Dense index (ChromaDB / sentence-transformers) ────────────────────────────
dense_collection = build_collection("helios_hybrid_dense", texts, metadatas, persist_dir="../.chroma")
print(f"Dense index: {dense_collection.count()} chunks")

# ── Sparse index (BM25) ───────────────────────────────────────────────────────
# Tokenise: lowercase, split on non-alphanumeric, keep part numbers intact
def tokenise(text: str) -> list[str]:
    # Keep hyphenated codes (HR-EE-GRIP-01, FW-V2-2.3.2) as single tokens
    text_lower = text.lower()
    tokens = re.findall(r'[a-z0-9]+(?:[\-\.][a-z0-9]+)*', text_lower)
    return tokens

tokenised_corpus = [tokenise(t) for t in texts]
bm25 = BM25Okapi(tokenised_corpus)
print(f"BM25 index: {len(tokenised_corpus)} documents")

# Preview tokenisation of a chunk containing part numbers
sample_idx = next(i for i, t in enumerate(texts) if 'HR-EE-GRIP-01' in t)
print(f"\nSample tokenisation (chunk with HR-EE-GRIP-01):")
sample_tokens = tokenised_corpus[sample_idx]
part_num_tokens = [t for t in sample_tokens if '-' in t]
print(f"  Part number tokens: {part_num_tokens[:10]}")

## Step 2 — Understand Reciprocal Rank Fusion (RRF)

RRF combines ranked lists without requiring normalised scores. For each document, it accumulates:

$$\text{RRF}(d) = \sum_{i} \frac{1}{k + \text{rank}_i(d)}$$

Where `k` is a smoothing constant (typically 60), and the sum is over all ranked lists. Documents appearing in multiple lists get higher scores. Documents that never appear get score 0.

**Key insight**: RRF only cares about *rank position*, not raw score values, so it cleanly merges very different scoring systems (cosine similarity vs BM25).

In [ ]:
from ragkit.vectorstore import query_collection, Hit

def bm25_search(query: str, top_k: int = 20) -> list[Hit]:
    """BM25 sparse retrieval — returns Hit objects."""
    query_tokens = tokenise(query)
    scores = bm25.get_scores(query_tokens)
    top_idx = scores.argsort()[::-1][:top_k]
    
    hits = []
    for rank, idx in enumerate(top_idx):
        if scores[idx] > 0:
            hits.append(Hit(
                text=texts[idx],
                metadata=metadatas[idx],
                score=float(scores[idx]),
                rank=rank + 1,
            ))
    return hits

def reciprocal_rank_fusion(
    *ranked_lists: list[Hit],
    k: int = 60,
    top_k: int = 5,
) -> list[Hit]:
    """
    Combine multiple ranked hit lists using Reciprocal Rank Fusion.
    Documents are identified by their text content.
    """
    scores: dict[str, float] = {}
    text_to_hit: dict[str, Hit] = {}
    
    for ranked_list in ranked_lists:
        for hit in ranked_list:
            doc_key = hit.text  # identity = text content
            rrf_score = 1.0 / (k + hit.rank)
            scores[doc_key] = scores.get(doc_key, 0) + rrf_score
            text_to_hit[doc_key] = hit
    
    sorted_keys = sorted(scores, key=lambda d: scores[d], reverse=True)
    result = []
    for new_rank, key in enumerate(sorted_keys[:top_k]):
        h = text_to_hit[key]
        result.append(Hit(text=h.text, metadata=h.metadata,
                          score=scores[key], rank=new_rank + 1))
    return result

print("Functions defined. Let's test on an exact-match query.")

## Step 3 — Where BM25 beats semantic search (exact codes)

In [ ]:
from ragkit.pretty import show_hits, compare_rankings

# Exact part number query — the part number string doesn't appear in many docs
exact_query = "What does HR-REED-UPGRADE fix?"

dense_hits = query_collection(dense_collection, exact_query, k=10)
bm25_hits  = bm25_search(exact_query, top_k=10)

print(f"Query: '{exact_query}'")
print()
compare_rankings(
    before=dense_hits[:5],
    after=bm25_hits[:5],
    title_before="Dense (semantic) top-5",
    title_after="BM25 (keyword) top-5",
)

# Check if HR-REED-UPGRADE appears in top results
print("\nDoes top dense hit contain 'HR-REED-UPGRADE'?",
      'HR-REED-UPGRADE' in dense_hits[0].text if dense_hits else False)
print("Does top BM25 hit contain 'HR-REED-UPGRADE'?",
      'HR-REED-UPGRADE' in bm25_hits[0].text if bm25_hits else False)

## Step 4 — Where semantic search beats BM25 (paraphrase)

In [ ]:
# Paraphrase query — words don't appear in documents but meaning does
paraphrase_q = "arm heat-related stopping problem"

dense_para = query_collection(dense_collection, paraphrase_q, k=5)
bm25_para  = bm25_search(paraphrase_q, top_k=5)

compare_rankings(
    before=dense_para,
    after=bm25_para[:5],
    title_before="Dense — paraphrase query",
    title_after="BM25 — paraphrase query",
)

print("\nDense finds the overheating document even with different words.")
print("BM25 might miss it because 'heat-related' ≠ 'thermal' or 'temperature'.")

## Step 5 — Hybrid with RRF: best of both worlds

In [ ]:
def hybrid_rag_dense_sparse(question: str, dense_collection, retrieve_k=20, final_k=5,
                             rrf_k=60, verbose=True):
    """Hybrid RAG: dense + BM25, fused with RRF, then generate."""
    # 1. Dense retrieval
    dense_hits = query_collection(dense_collection, question, k=retrieve_k)
    
    # 2. Sparse retrieval
    bm25_hits = bm25_search(question, top_k=retrieve_k)
    
    # 3. RRF fusion
    fused = reciprocal_rank_fusion(dense_hits, bm25_hits, k=rrf_k, top_k=final_k)
    
    if verbose:
        show_hits(fused, title=f"RRF-fused top-{final_k}")
    
    # 4. Generate
    from ragkit.llm import generate
    SYSTEM = """Technical assistant for Helios Robotics. Answer precisely using the context.
    Include part numbers, firmware versions, incident IDs when available."""
    context = "\n\n".join(f"[{h.metadata['source']}]\n{h.text}" for h in fused)
    answer = generate(f"Context:\n{context}\n\n---\nQuestion: {question}", system=SYSTEM)
    
    return answer, fused

# Test on the exact part number query
print(f"Query: '{exact_query}'")
answer, fused = hybrid_rag_dense_sparse(exact_query, dense_collection)
from ragkit.pretty import show_answer
show_answer(answer)

In [ ]:
# Run the benchmark suite comparing Naive vs Hybrid
benchmark_queries = [
    ("exact code",  "What does part number HR-REED-UPGRADE fix?"),
    ("exact code",  "Which firmware version is FW-V2-2.3.2 and what does it address?"),
    ("paraphrase",  "arm joint heat problem stopping production"),
    ("paraphrase",  "mobile robot energy estimate accuracy issue"),
]

from ragkit.llm import generate

for typ, q in benchmark_queries:
    print(f"\n{'='*60}")
    print(f"[{typ}] {q}")
    print()
    
    # Naive
    naive_hits = query_collection(dense_collection, q, k=5)
    naive_ctx = "\n\n".join(f"[{h.metadata['source']}]\n{h.text}" for h in naive_hits)
    naive_ans = generate(f"Context:\n{naive_ctx}\n---\nQuestion: {q}",
                         system="Answer using the provided context.")
    
    # Hybrid
    hybrid_ans, _ = hybrid_rag_dense_sparse(q, dense_collection, verbose=False)
    
    print(f"NAIVE:  {naive_ans[:180]}..." if len(naive_ans) > 180 else f"NAIVE:  {naive_ans}")
    print(f"HYBRID: {hybrid_ans[:180]}..." if len(hybrid_ans) > 180 else f"HYBRID: {hybrid_ans}")

## Step 6 — The RRF k parameter

`k=60` is the standard default, but let's see how it affects results.

In [ ]:
import matplotlib.pyplot as plt

# Show how RRF score changes with k for different rank positions
k_values = [1, 10, 30, 60, 120]
ranks = list(range(1, 21))

fig, ax = plt.subplots(figsize=(8, 4))
for k in k_values:
    scores = [1 / (k + r) for r in ranks]
    ax.plot(ranks, scores, marker='o', markersize=4, label=f'k={k}')

ax.set_xlabel('Rank position')
ax.set_ylabel('RRF score contribution')
ax.set_title('RRF score vs rank position for different k values', fontsize=11)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print("\nKey insight:")
print("  Low k  → scores are steep, rank 1 much better than rank 2 (aggressive)")
print("  High k → scores are flat, ranks 1-5 all similar (democratic)")
print("  k=60   → standard default, good balance")

## Step 7 — The infographic's Hybrid: vector + graph

The infographic's "Hybrid RAG" specifically means combining vector search with graph context. This is what we built in NB04 — here's a unified function that adds BM25 on top.

In [ ]:
if USE_NEO4J:
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(cfg.NEO4J_URI, auth=(cfg.NEO4J_USER, cfg.NEO4J_PASSWORD))
    try:
        driver.verify_connectivity()
        neo4j_available = True
        print("Neo4j connected")
    except Exception:
        neo4j_available = False
        print("Neo4j not available, skipping vector+graph section")
else:
    neo4j_available = False

In [ ]:
if neo4j_available:
    from ragkit.embeddings import embed, cosine_similarity
    import numpy as np

    def full_hybrid_rag(question: str, dense_collection, driver, retrieve_k=20, final_k=5):
        """
        Full hybrid: dense vector + BM25 + graph traversal.
        Three sources, fused with RRF for text then augmented with graph facts.
        """
        # Dense
        dense_hits = query_collection(dense_collection, question, k=retrieve_k)
        # BM25
        bm25_hits = bm25_search(question, top_k=retrieve_k)
        # RRF
        fused = reciprocal_rank_fusion(dense_hits, bm25_hits, k=60, top_k=final_k)
        
        # Graph: find seeds via embedded query, traverse
        q_vec = embed([question])[0].tolist()
        with driver.session() as session:
            try:
                seed_result = session.run("""
                    CALL db.index.vector.queryNodes('entity_embedding', 3, $vec)
                    YIELD node, score RETURN node.id AS id, score
                """, vec=q_vec)
                seed_ids = [r['id'] for r in seed_result]
            except Exception:
                seed_ids = []
        
        graph_facts = []
        for seed_id in seed_ids:
            with driver.session() as session:
                result = session.run("""
                    MATCH (a:Entity {id:$id})-[r]->(b:Entity)
                    RETURN a.name+' --['+type(r)+']--> '+b.name+': '+b.description AS fact
                    LIMIT 10
                """, id=seed_id)
                graph_facts.extend(r['fact'] for r in result)
        
        # Build prompt
        from ragkit.llm import generate
        text_ctx = "\n\n".join(f"[{h.metadata['source']}]\n{h.text}" for h in fused)
        graph_ctx = "\n".join(f"- {f}" for f in graph_facts[:12])
        
        prompt = f"""TEXT CONTEXT (dense+BM25 hybrid):
{text_ctx}

GRAPH CONTEXT (entity relationships):
{graph_ctx}

---
Question: {question}"""
        
        SYSTEM = "Answer using all three context sources. Be precise about part numbers and names."
        return generate(prompt, system=SYSTEM)

    # Test
    q = "Who is responsible for the firmware fix for Joint 4 and what part number does it involve?"
    ans = full_hybrid_rag(q, dense_collection, driver)
    from ragkit.pretty import show_answer
    show_answer(ans, title="Full Hybrid RAG (dense + BM25 + graph)")
else:
    print("Skipping — Neo4j not available")
    print("The vector+graph hybrid is covered in notebook 04.")

## Exercise — Fusing keyword and meaning with RRF (Korean grammar)

**BM25** matches exact tokens, so it shines on rare, literal terms (a specific particle like `은/는`). **Dense** search matches meaning, so it shines on paraphrases (*"marks the sentence topic"*). **Reciprocal Rank Fusion (RRF)** merges two ranked lists so a document that scores well in *either* method floats to the top.

Three candidate docs: `d_topic` (the topic particle — the true answer), `d_subject`, `d_object`.

1. **Read the rankings**: BM25 (for an exact-token query) and dense (for a paraphrase) are given as ordered lists.
2. **Implement**: Complete `rrf` using the standard formula `1 / (k + rank)`.
3. **Check**: The fused top-1 should be `d_topic` — even though neither method is perfect alone.
4. **Reflect**: One sentence — what happens to fusion as `k` grows very large?

In [ ]:
# Two retrievers, two ranked lists of document ids (rank 0 = best).
bm25_rank  = ["d_topic", "d_object", "d_subject"]   # exact-token query: "은/는 particle"
dense_rank = ["d_topic", "d_subject", "d_object"]   # paraphrase: "marks the sentence topic"

# ── Task 2: Reciprocal Rank Fusion ────────────────────────────────────────────
def rrf(rankings: list[list[str]], k: int = 60) -> list[str]:
    scores: dict[str, float] = {}
    for ranking in rankings:
        for rank, doc in enumerate(ranking):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

fused = rrf([bm25_rank, dense_rank])
print("Fused order:", fused)

# A harder case: the two retrievers disagree on #1, but d_topic is high in both.
fused_disagree = rrf([["d_object", "d_topic", "d_subject"],
                      ["d_topic", "d_subject", "d_object"]])
print("Fused (disagreement):", fused_disagree)

# ── Task 4: effect of large k (comment) ───────────────────────────────────────
#   Your answer:

# ── Self-check ────────────────────────────────────────────────────────────────
assert fused[0] == "d_topic", "the doc ranked well by both methods should win"
assert fused_disagree[0] == "d_topic", "appearing high in both lists should beat a single #1"
print("\n✅ Exercise checks passed!")

## Tradeoffs

| Aspect | Dense Only | BM25 Only | Dense + BM25 (Hybrid) | + Graph |
|---|---|---|---|---|
| **Semantic understanding** | ★★★★★ | ★★☆☆☆ | ★★★★★ | ★★★★★ |
| **Exact string/code match** | ★★☆☆☆ | ★★★★★ | ★★★★★ | ★★★★★ |
| **Multi-hop** | ★☆☆☆☆ | ★☆☆☆☆ | ★☆☆☆☆ | ★★★★★ |
| **Setup complexity** | ★☆☆☆☆ | ★☆☆☆☆ | ★★☆☆☆ | ★★★★☆ |
| **Latency** | ★★★★☆ | ★★★★★ | ★★★★☆ | ★★★☆☆ |
| **When to use** | Paraphrase-heavy | Code/ID lookup | General production | Complex domains |

**Practical recommendation**: Dense + BM25 + RRF is the most widely adopted production pattern. It's easy to implement, robust, and handles the majority of real-world retrieval failures.

## Exercises

1. **Tune RRF k**: Try `k=1`, `k=10`, `k=100`. On which queries does the optimal k differ?
2. **Weighted RRF**: Modify `reciprocal_rank_fusion` to accept per-list weights. Can you find a query where upweighting BM25 helps?
3. **Add a third retriever**: Add a simple `TF-IDF` retriever (sklearn `TfidfVectorizer`) alongside BM25. Does a three-way RRF fusion help?
4. **ChromaDB with hybrid**: ChromaDB supports `where` filters. Combine semantic search with a metadata filter (e.g., only search `category='spec'`). Is this better for spec-only queries?

**Next:** [06_agentic_rag_router.ipynb](06_agentic_rag_router.ipynb) — let an LLM agent decide how to retrieve.